In [1]:
# Tải mã nguồn LLaMA Factory bản mới nhất từ GitHub
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory

# Cài đặt các thư viện bắt buộc cho quá trình huấn luyện và tính toán metrics
!pip install -e .[torch,metrics]

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 672, done.
remote: Counting objects: 100% (672/672), done.
remote: Compressing objects: 100% (507/507), done.
remote: Total 672 (delta 156), reused 435 (delta 103), pack-reused 0 (from 0)
Receiving objects: 100% (672/672), 5.36 MiB | 20.16 MiB/s, done.
Resolving deltas: 100% (156/156), done.
/kaggle/working/LLaMA-Factory
Obtaining file:///kaggle/working/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 9.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 47.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 22.

In [2]:
# 1. Tạo thư mục đích trong LLaMA Factory
!mkdir -p data/litepp_rco

# 2. Tìm kiếm bất chấp tên thư mục và copy tất cả file json từ Input sang
!find /kaggle/input/ -name "*.json" -exec cp {} data/litepp_rco/ \;

# 3. Ghi đè cấu hình ánh xạ dataset_info.json bằng Python cho mượt mà
import json
dataset_info = {
  "litepp_train": { "file_name": "train.json" },
  "litepp_val": { "file_name": "val.json" }
}
with open("data/litepp_rco/dataset_info.json", "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

# 4. Kiểm tra trực quan xem file đã vào vị trí chưa
print("--- KIỂM TRA ĐƯỜNG DẪN FILE THỰC TẾ ---")
!ls -l data/litepp_rco/

--- KIỂM TRA ĐƯỜNG DẪN FILE THỰC TẾ ---
total 2728
-rw-r--r-- 1 root root     108 Jun 17 13:58 dataset_info.json
-rw-r--r-- 1 root root 2503933 Jun 17 13:58 train.json
-rw-r--r-- 1 root root  281597 Jun 17 13:58 val.json


In [3]:
%%writefile rco_config.yaml
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
stage: sft
do_train: true
finetuning_type: lora
lora_target: all
lora_rank: 8
lora_alpha: 16
lora_dropout: 0.0

dataset: litepp_train
eval_dataset: litepp_val
dataset_dir: data/litepp_rco
template: qwen
cutoff_len: 4096

learning_rate: 2e-4
num_train_epochs: 3.0
per_device_train_batch_size: 1  
per_device_eval_batch_size: 1
gradient_accumulation_steps: 8      
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true

output_dir: saves/Qwen2.5-1.5B-RCO-LoRA
logging_steps: 10
save_steps: 100
eval_steps: 100
eval_strategy: steps
load_best_model_at_end: true

ddp_find_unused_parameters: false   # bắt buộc với LoRA multi-GPU

Writing rco_config.yaml


In [4]:
import os
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"

!torchrun \
    --nproc_per_node 2 \
    --master_port 29500 \
    /kaggle/working/LLaMA-Factory/src/llamafactory/launcher.py \
    rco_config.yaml

W0617 13:58:43.290000 151 torch/distributed/run.py:852] 
W0617 13:58:43.290000 151 torch/distributed/run.py:852] *****************************************
W0617 13:58:43.290000 151 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0617 13:58:43.290000 151 torch/distributed/run.py:852] *****************************************
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_d

In [5]:
# Gọi CLI của LLaMA Factory để bắt đầu quá trình SFT Finetuning
#!llamafactory-cli train rco_config.yaml

In [6]:
!ls -lh saves/Qwen2.5-1.5B-RCO-LoRA

total 47M
-rw-r--r-- 1 root root 1.1K Jun 17 14:17 adapter_config.json
-rw-r--r-- 1 root root  36M Jun 17 14:17 adapter_model.safetensors
-rw-r--r-- 1 root root  347 Jun 17 14:18 all_results.json
-rw-r--r-- 1 root root 2.5K Jun 17 14:17 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Jun 17 14:15 checkpoint-100
drwxr-xr-x 2 root root 4.0K Jun 17 14:17 checkpoint-114
-rw-r--r-- 1 root root  160 Jun 17 14:18 eval_results.json
-rw-r--r-- 1 root root 1.8K Jun 17 14:18 README.md
-rw-r--r-- 1 root root  692 Jun 17 14:17 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun 17 14:17 tokenizer.json
-rw-r--r-- 1 root root 2.6K Jun 17 14:17 trainer_log.jsonl
-rw-r--r-- 1 root root 3.3K Jun 17 14:17 trainer_state.json
-rw-r--r-- 1 root root 5.8K Jun 17 14:17 training_args.bin
-rw-r--r-- 1 root root  207 Jun 17 14:17 train_results.json


In [7]:
# 1. Quay trở ra thư mục làm việc gốc của Kaggle để đảm bảo an toàn quyền ghi file
%cd /kaggle/working/

# 2. Sử dụng đường dẫn tuyệt đối để nén thư mục chứa model LoRA vừa train xong
!zip -r qwen_lora_rco.zip LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA

# 3. Tạo link download trực tiếp trên giao diện Notebook Kaggle
from IPython.display import FileLink
FileLink(r'qwen_lora_rco.zip')

/kaggle/working
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA/ (stored 0%)
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA/adapter_model.safetensors (deflated 7%)
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA/adapter_config.json (deflated 58%)
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA/train_results.json (deflated 38%)
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA/chat_template.jinja (deflated 71%)
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA/trainer_log.jsonl (deflated 74%)
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA/eval_results.json (deflated 33%)
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA/all_results.json (deflated 51%)
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA/tokenizer.json (deflated 81%)
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA/checkpoint-114/ (stored 0%)
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-RCO-LoRA/checkpoint-114/adapter_model.safetensors (deflated 7%)
  adding: LLaMA-Factory/saves/Qwen2.5-1.5B-R

/kaggle/working/qwen_lora_rco.zip